# GPT-2: Generating Text

GPT-2 is an autoregressive, decoder-only Transformer released by OpenAI in 2019. It generates text one token at a time, using the preceding tokens to predict what comes next.

Unlike the instruction-tuned models used in the prompting notebook, this base model was trained to continue text rather than act as a conversational assistant.

We provide a short, everyday prompt and compare multiple continuations at different temperatures. This lets us examine how sampling changes the variety and consistency of the generated text.


## Sampling at a higher temperature

We generate three continuations for the same prompt using `temperature=0.7`. Compared with a lower temperature, this produces a less concentrated next-token distribution, allowing more variation when sampling.

The model may generate fluent text that is repetitive, inconsistent, or factually incorrect.


In [1]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

tokenizer = AutoTokenizer.from_pretrained("gpt2")
model = AutoModelForCausalLM.from_pretrained("gpt2")
model.eval()

prompt = "She peeled an orange and noticed"
inputs = tokenizer(prompt, return_tensors="pt")

torch.manual_seed(42)

with torch.inference_mode():
    outputs = model.generate(
        **inputs,
        max_new_tokens=50,
        do_sample=True,
        num_return_sequences=3,
        temperature=0.7,
        top_k=0,
        top_p=1.0,
        pad_token_id=tokenizer.eos_token_id,
    )

for i, output in enumerate(outputs, 1):
    print(f"\nModel output {i}:")
    print(tokenizer.decode(output, skip_special_tokens=True))


/home/nick/github-projects/prompting-base-models/venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 148/148 [00:00<00:00, 5000.66it/s]



Model output 1:
She peeled an orange and noticed her work in front of her was not as much as she expected. "I need to go to work," she said.

She left with her bare hands, waiting for her to drop them to her head to make sure she was OK.

Model output 2:
She peeled an orange and noticed that the girl was wearing a black hoodie. She rolled her eyes and smacked the girl on the head. "I'll be back by noon next week."

The girl said she wanted to go home and not be with her boyfriend.

Model output 3:
She peeled an orange and noticed the red. She had been surprised to find that there were two crows in the grass.

"What does that mean?" Jaune asked.

"I don't know, but it means something to her," the man replied. When


### About the output

Each output starts with the same prompt, but the sampled continuation can differ. At `temperature=0.7`, the model can select from a broader range of tokens than at a lower temperature.

GPT-2 is continuing text, not verifying facts or following a conversation. Read the outputs for variation, coherence, and repetition.

Next, we repeat the prompt with `temperature=0.1`, keeping the other generation settings unchanged. A lower temperature concentrates probability on the most likely next tokens. Sampling remains enabled, so identical outputs are not guaranteed.


In [2]:
torch.manual_seed(42)

with torch.inference_mode():
    outputs = model.generate(
        **inputs,
        max_new_tokens=50,
        do_sample=True,
        num_return_sequences=3,
        temperature=0.1,
        top_k=0,
        top_p=1.0,
        pad_token_id=tokenizer.eos_token_id,
    )

for i, output in enumerate(outputs, 1):
    print(f"\nModel output {i}:")
    print(tokenizer.decode(output, skip_special_tokens=True))



Model output 1:
She peeled an orange and noticed that the red was still there.

"I'm sorry, I'm sorry," she said. "I'm sorry. I'm sorry. I'm sorry. I'm sorry. I'm sorry. I'm sorry. I'm sorry.

Model output 2:
She peeled an orange and noticed that the red was still there.

"I'm sorry, I'm sorry," she said. "I'm sorry. I'm sorry. I'm sorry. I'm sorry. I'm sorry. I'm sorry. I'm sorry.

Model output 3:
She peeled an orange and noticed that the red was still there.

"I'm sorry, but I'm not sure what to do," she said. "I'm not sure what to do."

She was told to go to the bathroom and get dressed.



### Comparing the outputs

Compare the three continuations at each temperature. At `temperature=0.1`, probability is more concentrated on high-probability tokens, so the continuations may be more similar. At `temperature=0.7`, sampling has more room to select alternatives.

Both examples use sampling. A lower temperature does not turn sampling into greedy decoding or guarantee identical outputs. The random seed is reset before each run to make the comparison easier to repeat within the same environment.

Temperature changes the sampling distribution, not the model’s training or factual reliability. A base model can produce coherent text, but neither setting guarantees a coherent or accurate continuation. Instruction tuning is a separate process that trains a model to respond to instructions.
